# 🔧 02 — Prepare Scenes
### Amazon Deforestation Change Detection Pipeline

Loads raw Sentinel-2 bands, normalises pixel values, and computes NDVI.
Outputs aligned GeoTIFFs ready for DOFA inference.

---
**Pipeline:**
1. ✅ `01_download_data.ipynb`
2. 🔧 `02_prepare_scenes.ipynb` ← *you are here*
3. 🌳 `03_detect_changes.ipynb`
4. 🎨 `04_visualise.ipynb`


## ⚙️ Configuration

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

RAW_DIR      = Path("data/raw")
PREPARED_DIR = Path("data/prepared")
YEARS        = ["2019", "2024"]
TARGET_CRS   = "EPSG:32720"   # UTM zone 20S — covers Rondônia

PREPARED_DIR.mkdir(parents=True, exist_ok=True)
print("✅ Configuration set")


## 🔍 Step 1 — Find Band Files

In [ ]:
def find_band_file(year_dir, band):
    patterns = [
        f"*_{band}_10m.jp2", f"*_{band}.jp2",
        f"*_{band}_10m.tif", f"*_{band}.tif", f"{band}.tif",
    ]
    for pattern in patterns:
        matches = list(year_dir.rglob(pattern))
        if matches:
            return matches[0]
    return None

# Check what's available
for year in YEARS:
    year_dir = RAW_DIR / year
    print(f"\n📁 {year}:")
    for band in ["B04", "B03", "B02", "B08"]:
        f = find_band_file(year_dir, band)
        print(f"   {band}: {'✅ ' + str(f.name) if f else '❌ Not found'}")


## 📐 Step 2 — Load, Normalise & Stack Bands

In [ ]:
def load_band(year, band_name):
    year_dir  = RAW_DIR / year
    band_file = find_band_file(year_dir, band_name)
    if band_file is None:
        raise FileNotFoundError(f"Band {band_name} not found in {year_dir}")
    with rasterio.open(band_file) as src:
        return src.read(1).astype(np.float32), src.profile, src.transform, src.crs

def normalise(band, p_low=2, p_high=98):
    p2, p98 = np.percentile(band, p_low), np.percentile(band, p_high)
    return np.clip((band - p2) / (p98 - p2 + 1e-6), 0, 1)

def compute_ndvi(red, nir):
    return np.clip((nir - red) / (nir + red + 1e-6), -1, 1)

def save_geotiff(data, path, profile):
    if data.ndim == 2:
        data = data[np.newaxis, :]
    profile.update(count=data.shape[0], dtype="float32", driver="GTiff", compress="lzw")
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(data.astype(np.float32))
    print(f"   Saved: {path}  shape={data.shape}")

def prepare_year(year):
    print(f"\n📅 Preparing {year}...")
    try:
        red,   profile, transform, crs = load_band(year, "B04")
        green, *_ = load_band(year, "B03")
        blue,  *_ = load_band(year, "B02")
        nir,   *_ = load_band(year, "B08")
    except FileNotFoundError as e:
        print(f"   ⚠️  {e}\n   → Creating synthetic scene for demo...")
        return create_synthetic_scene(year)

    rgb  = np.stack([normalise(red), normalise(green), normalise(blue)], axis=0)
    ndvi = compute_ndvi(red, nir)
    profile.update(crs=crs, transform=transform)
    save_geotiff(rgb,  PREPARED_DIR / f"{year}_rgb.tif",  profile)
    save_geotiff(ndvi, PREPARED_DIR / f"{year}_ndvi.tif", profile)
    print(f"   ✅ Done — NDVI range: [{ndvi.min():.3f}, {ndvi.max():.3f}]")
    return rgb, ndvi

def create_synthetic_scene(year):
    np.random.seed(42 if year == "2019" else 99)
    H, W = 512, 512
    if year == "2019":
        r = np.random.normal(0.08, 0.02, (H,W)).clip(0,1).astype(np.float32)
        g = np.random.normal(0.15, 0.03, (H,W)).clip(0,1).astype(np.float32)
        b = np.random.normal(0.07, 0.02, (H,W)).clip(0,1).astype(np.float32)
        ndvi = np.random.normal(0.75, 0.08, (H,W)).clip(-1,1).astype(np.float32)
    else:
        r = np.random.normal(0.18, 0.05, (H,W)).clip(0,1).astype(np.float32)
        g = np.random.normal(0.18, 0.04, (H,W)).clip(0,1).astype(np.float32)
        b = np.random.normal(0.12, 0.03, (H,W)).clip(0,1).astype(np.float32)
        ndvi = np.random.normal(0.45, 0.15, (H,W)).clip(-1,1).astype(np.float32)
        for i in range(0, W, 40):
            r[:, i:i+8] = 0.35; g[:, i:i+8] = 0.30; ndvi[:, i:i+8] = 0.05
    rgb = np.stack([r, g, b], axis=0)
    transform = from_bounds(-63.5, -11.5, -62.5, -10.5, W, H)
    profile = {"driver":"GTiff","dtype":"float32","width":W,"height":H,"count":3,
               "crs":"EPSG:4326","transform":transform}
    save_geotiff(rgb,  PREPARED_DIR / f"{year}_rgb.tif",  profile)
    save_geotiff(ndvi, PREPARED_DIR / f"{year}_ndvi.tif", profile)
    print(f"   ⚠️  Synthetic data created. Replace with real Sentinel-2 for actual analysis.")
    return rgb, ndvi

for year in YEARS:
    prepare_year(year)


## 🔎 Step 3 — Verify Alignment

In [ ]:
shapes = {}
for year in YEARS:
    path = PREPARED_DIR / f"{year}_rgb.tif"
    with rasterio.open(path) as src:
        shapes[year] = (src.count, src.height, src.width)
        print(f"{year}: shape={shapes[year]}  crs={src.crs}  bounds={src.bounds}")

if list(shapes.values())[0] == list(shapes.values())[1]:
    print("\n✅ Scenes aligned — ready for change detection")
else:
    print("\n⚠️  Shape mismatch — will be cropped to minimum size in next step")


## 🖼️ Quick Preview

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("Prepared Scenes Preview", fontsize=14, fontweight="bold")

for row, year in enumerate(YEARS):
    with rasterio.open(PREPARED_DIR / f"{year}_rgb.tif") as src:
        rgb = src.read()
    with rasterio.open(PREPARED_DIR / f"{year}_ndvi.tif") as src:
        ndvi = src.read(1)

    axes[row, 0].imshow(np.transpose(rgb, (1, 2, 0)).clip(0, 1))
    axes[row, 0].set_title(f"RGB {year}", fontweight="bold")
    axes[row, 0].axis("off")

    im = axes[row, 1].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=1.0)
    axes[row, 1].set_title(f"NDVI {year}  (mean={ndvi.mean():.3f})", fontweight="bold")
    axes[row, 1].axis("off")
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046)

plt.tight_layout()
plt.savefig(PREPARED_DIR / "preview.png", dpi=120, bbox_inches="tight")
plt.show()
print("\n✅ Proceed to: 03_detect_changes.ipynb")
